# 02. Frozen baselines before training

The old prompt contained numbered example mappings. The new prompt omits those examples,
supplies bubble/character coordinates and OCR, and says IDs are arbitrary. Measure the frozen
model first to distinguish prompt/image improvements from training improvements.

Compare page-only versus the original side legend, higher visual resolution and text-only input.
Use val if available; a smoke-page result is just a diagnostic. Do not inspect test pages here.
All final LoRA comparisons in 07 repeat the frozen baseline at the same image settings.


In [ ]:
from pathlib import Path
import sys
import json
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "models.json").exists():
    ROOT = ROOT.parent
if not (ROOT / "models.json").exists():
    raise FileNotFoundError("Start the kernel in 1pageTest or its 6_lora_training folder.")
sys.path.insert(0, str(ROOT / "6_lora_training"))
import lora_helpers as H
print("Interpreter:", sys.executable)
print("Dataset:", H.PAGE_DIR)
print("Local base model:", H.MODEL_DIR)


In [ ]:
pages = H.read_pages()
split = "val" if any(p["split"] == "val" for p in pages) else "smoke"
evaluation_pages = H.partition(pages, split)
print("Evaluating", split, "— smoke results do not measure generalization.")
settings = [
    {"name": "frozen_page_one_tile", "view": "page_only", "max_tiles": 1},
    {"name": "frozen_page_four_tiles", "view": "page_only", "max_tiles": 4},
    {"name": "frozen_with_legend", "view": "annotated", "max_tiles": 1},
    {"name": "frozen_text_only", "view": "page_only", "max_tiles": 1, "text_only": True},
]
model, tokenizer = H.load_base()
try:
    results = [H.evaluate_model(model, tokenizer, evaluation_pages, setting, setting["name"],
                 H.RUN_DIR / "baselines" / f"{setting['name']}_{split}.json")
               for setting in settings]
    display([result["summary"] for result in results])
finally:
    del model, tokenizer
    H.cleanup()
